## data prep 

In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
from pathlib import Path

In [5]:
# Flexible file path handling
candidate_paths = [
    Path("laptop_data.csv"),
    Path("data/laptop_data.csv"),
    Path("../data/laptop_data.csv"),
    Path("/mnt/data/laptop_data.csv")
]

data_path = None
for p in candidate_paths:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not find laptop_data.csv. Place it in the notebook folder, data/, or ../data/.")

print("Using data file:", data_path)
df = pd.read_csv(data_path)
print("Shape:", df.shape)
df.head()

FileNotFoundError: Could not find laptop_data.csv. Place it in the notebook folder, data/, or ../data/.

In [ ]:
print("Dtypes:")
display(df.dtypes)

print("\nMissing values (top 20):")
display(df.isnull().sum().sort_values(ascending=False).head(53))

Dtypes:


NameError: name 'df' is not defined

In [4]:
data = df.copy()

In [5]:
# Drop leakage and weak raw-text columns
drop_cols = [
    "scenario_id",
    "base_laptop_id",
    "label_reason",
    "cpu_raw",
    "gpu_raw",
    "memory_raw",
    "screen_resolution_raw"
]

data = data.drop(columns=[c for c in drop_cols if c in data.columns], errors="ignore")
print("After dropping leakage/raw columns:", data.shape)

After dropping leakage/raw columns: (19545, 46)


In [ ]:
#avoid bias/noise 
data = data.drop(columns=["company", "price_tier"], errors="ignore")
print("After dropping company and price_tier:", data.shape)

After dropping company and price_tier: (19545, 44)


In [ ]:
def clean_numeric(series):
    return (
        series.astype(str)
        .str.replace(r"[^\d.\-]", "", regex=True)
        .replace("", np.nan)
        .astype(float)
    )

numeric_string_cols = [
    "price",
    "recommended_upgrade_cost_est",
    "ram_upgrade_cost_est",
    "storage_upgrade_cost_est",
    "gpu_upgrade_cost_est"
]

for col in numeric_string_cols:
    if col in data.columns:
        data[col] = clean_numeric(data[col])

print(data[numeric_string_cols].dtypes)

price                           float64
recommended_upgrade_cost_est    float64
ram_upgrade_cost_est            float64
storage_upgrade_cost_est        float64
gpu_upgrade_cost_est            float64
dtype: object


In [ ]:
for col in numeric_string_cols:
    if col in data.columns:
        data[col] = data[col].fillna(data[col].median())

print("Nulls after fill:")
display(data[numeric_string_cols].isnull().sum())

Nulls after fill:


price                           0
recommended_upgrade_cost_est    0
ram_upgrade_cost_est            0
storage_upgrade_cost_est        0
gpu_upgrade_cost_est            0
dtype: int64

In [9]:
# Feature engineering
data["performance_score"] = (
    data["cpu_score"] * (1 + np.log1p(data["ram_gb"])) +
    data["gpu_score"] * 0.8
)

data["ram_gap_ratio"] = data["ram_gap_gb"] / (data["target_ram_gb"] + 1)
data["storage_gap_ratio"] = data["storage_gap_gb"] / (data["target_storage_gb"] + 1)
data["gpu_gap_ratio"] = data["gpu_gap_score"] / (data["target_gpu_score"] + 1)

data["upgrade_pressure"] = (
    0.35 * data["ram_gap_ratio"] +
    0.25 * data["storage_gap_ratio"] +
    0.40 * data["gpu_gap_ratio"]
)

data["total_urgency"] = (
    data["ram_urgency_score"] +
    data["storage_urgency_score"] +
    data["gpu_urgency_score"]
) / 3

data["price_per_performance"] = data["price"] / (data["performance_score"] + 1)

data["storage_capacity_score"] = (
    data["ssd_gb"] +
    data["hdd_gb"] +
    data["flash_gb"] +
    data["hybrid_gb"]
)

data["storage_speed_score"] = (
    1.00 * data["ssd_gb"] +
    0.45 * data["hdd_gb"] +
    0.75 * data["flash_gb"] +
    0.65 * data["hybrid_gb"]
)

data[[
    "performance_score",
    "ram_gap_ratio",
    "storage_gap_ratio",
    "gpu_gap_ratio",
    "upgrade_pressure",
    "total_urgency",
    "price_per_performance",
    "storage_capacity_score",
    "storage_speed_score"
]].head()

,performance_score,ram_gap_ratio,storage_gap_ratio,gpu_gap_ratio,upgrade_pressure,total_urgency,price_per_performance,storage_capacity_score,storage_speed_score
0,12.150286,0.0,0.498054,0.0,0.124514,0.113667,5427.918441,128,128.0
1,12.150286,0.0,0.498054,0.0,0.124514,0.113667,5427.918441,128,128.0
2,12.150286,0.0,0.498054,0.0,0.124514,0.113667,5427.918441,128,128.0
3,12.150286,0.0,0.498054,0.0,0.124514,0.113667,5427.918441,128,128.0
4,12.150286,0.0,0.498054,0.0,0.124514,0.113667,5427.918441,128,128.0


In [10]:
print("Final dtypes:")
display(data.dtypes)

print("\nTarget distribution: upgrade_first")
display(data["upgrade_first"].value_counts())

print("\nCleaned shape:")
print(data.shape)

Final dtypes:


type_name                           str
user_profile                        str
budget_class                        str
upgrade_first                       str
gain_class                          str
price                           float64
recommended_upgrade_cost_est    float64
inches                          float64
weight_kg                       float64
opsys                               str
ram_gb                            int64
ssd_gb                            int64
hdd_gb                            int64
flash_gb                          int64
hybrid_gb                         int64
total_storage_gb                  int64
has_ssd                           int64
has_hdd                           int64
screen_width                      int64
screen_height                     int64
is_ips                            int64
is_touchscreen                    int64
is_retina                         int64
is_4k                             int64
cpu_brand                           str



Target distribution: upgrade_first


upgrade_first
RAM                  8190
No Upgrade Needed    5659
Storage              3524
GPU                  2172
Name: count, dtype: int64


Cleaned shape:
(19545, 53)


In [11]:
# Save cleaned dataset
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

cleaned_path = output_dir / "cleaned_laptop_data.csv"
data.to_csv(cleaned_path, index=False)

print("Saved cleaned dataset to:", cleaned_path.resolve())

Saved cleaned dataset to: C:\Users\ahmad\Desktop\uni\3rd year\2nd sem\applied\laptop_decision_portal\preprocessing\data\cleaned_laptop_data.csv
